# 02 — Deploy as Hosted Agent

Packages the single skill-driven agent into a container and deploys it as a Microsoft Foundry **Hosted Agent**.

The deployed agent receives a `NormalizedDetection` JSON via the Responses API and returns a structured `InvestigationVerdict`.

### Lifecycle
1. **Package** — copy `single_agent_workflow.py`, `tools.py`, `subprocess_script_runner.py`, `skills/`, and `test_cases/` into `hosted_agent/`
2. **Build** — `docker build --platform linux/amd64`
3. **Push** — Azure Container Registry
4. **Deploy** — `project.agents.create_version()`
5. **Invoke** — send a detection via the Responses API

### Platform-injected env vars
| Variable | Description |
|---|---|
| `FOUNDRY_PROJECT_ENDPOINT` | Project endpoint (auto-injected) |
| `MODEL_DEPLOYMENT_NAME` | Deployment used by the agent (set at create_version time) |

> **Prerequisite**: Run `00-setup.ipynb` first. Docker Desktop must be running.


In [1]:
import json, os
from pathlib import Path

config_path = Path("../workshop_config.json")
if not config_path.exists():
    raise FileNotFoundError("../workshop_config.json not found. Run 00-setup.ipynb first.")
with open(config_path) as f:
    config = json.load(f)

RESOURCE_GROUP        = config["RESOURCE_GROUP"]
ACCOUNT_NAME          = config["ACCOUNT_NAME"]
PROJECT_NAME          = config["PROJECT_NAME"]
MODEL_DEPLOYMENT_NAME = config["MODEL_DEPLOYMENT_NAME"]
PROJECT_ENDPOINT      = config["PROJECT_ENDPOINT"]

from azure.identity import AzureCliCredential
credential = AzureCliCredential()
print(f"✅ Config loaded — deploying to {ACCOUNT_NAME}/{PROJECT_NAME}")


✅ Config loaded — deploying to fndryiac2ttkx3/iac-ops-demo


## Generate Deployment Artifacts

In [2]:
import shutil

HOSTED = Path("hosted_agent")
HOSTED.mkdir(exist_ok=True)

# 1) Python modules the container imports at startup.
for fname in ("single_agent_workflow.py", "tools.py", "subprocess_script_runner.py"):
    src = Path(fname)
    if not src.is_file():
        raise FileNotFoundError(f"Expected {src} alongside this notebook.")
    shutil.copy2(src, HOSTED / fname)
    print(f"  ✅ Synced: {HOSTED / fname}")

# 2) Skill directory tree (SKILL.md + references/ + scripts/).
src_skills = Path("skills")
dst_skills = HOSTED / "skills"
if dst_skills.exists():
    shutil.rmtree(dst_skills)
shutil.copytree(src_skills, dst_skills)
skill_files = sum(1 for _ in dst_skills.rglob("*") if _.is_file())
print(f"  ✅ Synced skills/ ({skill_files} files)")

# 3) Test-case scenarios (so a deployed agent can answer detections for the bundled UPNs).
src_cases = Path("test_cases")
dst_cases = HOSTED / "test_cases"
if dst_cases.exists():
    shutil.rmtree(dst_cases)
dst_cases.mkdir(parents=True, exist_ok=True)
case_count = 0
for case_file in sorted(src_cases.glob("*.json")):
    shutil.copy2(case_file, dst_cases / case_file.name)
    case_count += 1
print(f"  ✅ Bundled {case_count} test case(s)")

# 4) Sanity check that the committed hosted_agent/{main.py,Dockerfile,requirements.txt} exist.
for required in ("main.py", "Dockerfile", "requirements.txt"):
    p = HOSTED / required
    if not p.exists():
        raise FileNotFoundError(f"Expected {p} to exist (committed to the repo).")
    print(f"  ✓ Present: {p}")

# Enforce store=True so Foundry persists session/thread metadata.
hosted_main = HOSTED / "main.py"
main_text = hosted_main.read_text(encoding="utf-8")
if "store=False" in main_text:
    hosted_main.write_text(main_text.replace("store=False", "store=True"), encoding="utf-8")
    print("  ✅ Updated hosted_agent/main.py to store=True for portal session visibility")
elif "store=True" in main_text:
    print("  ✓ hosted_agent/main.py already uses store=True")

print("\n📁 Deployment artifacts ready in hosted_agent/")


  ✅ Synced: hosted_agent\single_agent_workflow.py
  ✅ Synced: hosted_agent\tools.py
  ✅ Synced: hosted_agent\subprocess_script_runner.py
  ✅ Synced skills/ (17 files)
  ✅ Bundled 5 test case(s)
  ✓ Present: hosted_agent\main.py
  ✓ Present: hosted_agent\Dockerfile
  ✓ Present: hosted_agent\requirements.txt

📁 Deployment artifacts ready in hosted_agent/


## Build, Push & Deploy

In [3]:
import asyncio, subprocess, sys

AGENT_NAME = "impossible-travel-skill-agent"
IMAGE_TAG  = "v1"
ACR_LOGIN_SERVER = ""   # Override if auto-discovery can't find your ACR.

_USE_SHELL = sys.platform == "win32"

def _run_az(args):
    r = subprocess.run(["az", *args], capture_output=True, text=True, shell=_USE_SHELL)
    if r.returncode != 0:
        print(f"  ⚠️  az {' '.join(args)} (exit {r.returncode}) {r.stderr.strip().splitlines()[0][:200] if r.stderr.strip() else ''}")
        return ""
    return r.stdout.strip()

if subprocess.run(["docker", "info"], capture_output=True, text=True, shell=_USE_SHELL).returncode != 0:
    raise RuntimeError("Docker daemon not reachable. Start Docker Desktop and retry.")

print("▶ Step 1: Discovering Azure Container Registry...")
if not ACR_LOGIN_SERVER:
    acr_id = _run_az(["cognitiveservices", "account", "show", "--name", ACCOUNT_NAME,
                       "--resource-group", RESOURCE_GROUP, "--query", "properties.containerRegistry", "-o", "tsv"])
    if acr_id and acr_id != "None":
        acr_name = acr_id.rsplit("/", 1)[-1]
        ACR_LOGIN_SERVER = _run_az(["acr", "show", "--name", acr_name, "--query", "loginServer", "-o", "tsv"])
if not ACR_LOGIN_SERVER:
    ACR_LOGIN_SERVER = _run_az(["acr", "list", "--resource-group", RESOURCE_GROUP, "--query", "[0].loginServer", "-o", "tsv"])
if not ACR_LOGIN_SERVER:
    ACR_LOGIN_SERVER = _run_az(["acr", "list", "--query", "[0].loginServer", "-o", "tsv"])
if not ACR_LOGIN_SERVER:
    raise RuntimeError("No ACR discovered. Set ACR_LOGIN_SERVER manually at the top of this cell.")
FULL_IMAGE = f"{ACR_LOGIN_SERVER}/{AGENT_NAME}:{IMAGE_TAG}"
print(f"  ACR:   {ACR_LOGIN_SERVER}")
print(f"  Image: {FULL_IMAGE}")

print("\n▶ Step 2: Building Docker image...")
subprocess.run(["docker", "build", "--platform", "linux/amd64", "-t", f"{AGENT_NAME}:{IMAGE_TAG}", "hosted_agent/"], check=True, shell=_USE_SHELL)

print("\n▶ Step 3: Pushing to ACR...")
acr_name = ACR_LOGIN_SERVER.split(".")[0]
subprocess.run(["az", "acr", "login", "--name", acr_name], check=True, shell=_USE_SHELL)
subprocess.run(["docker", "tag", f"{AGENT_NAME}:{IMAGE_TAG}", FULL_IMAGE], check=True, shell=_USE_SHELL)
subprocess.run(["docker", "push", FULL_IMAGE], check=True, shell=_USE_SHELL)
print("  ✅ Image pushed")

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import HostedAgentDefinition, ProtocolVersionRecord, AgentProtocol
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential, allow_preview=True)

# ── Step 3.5: RBAC for image pull + model access (mirrors main repo's deploy notebook). ──
print("\n▶ Step 3.5: Ensuring required RBAC role assignments...")
_acr_resource_id    = _run_az(["acr", "show", "--name", acr_name, "--query", "id", "-o", "tsv"])
_foundry_account_id = _run_az(["cognitiveservices", "account", "show", "--name", ACCOUNT_NAME,
                                "--resource-group", RESOURCE_GROUP, "--query", "id", "-o", "tsv"])
_project_principal  = _run_az(["resource", "show", "--ids",
                                f"{_foundry_account_id}/projects/{PROJECT_NAME}",
                                "--query", "identity.principalId", "-o", "tsv"])

def _ensure_role(principal_id, role, scope, label):
    if not principal_id or principal_id == "None":
        print(f"  ⚠️  Skipping '{role}' for {label}: no principal id resolved.")
        return
    existing = _run_az(["role", "assignment", "list", "--assignee", principal_id, "--scope", scope,
                         "--query", f"[?roleDefinitionName=='{role}'] | length(@)", "-o", "tsv"])
    if existing and existing != "0":
        print(f"  ✓ '{role}' already granted to {label}")
        return
    print(f"  • Granting '{role}' to {label} on {scope.split('/')[-1]}")
    _run_az(["role", "assignment", "create", "--assignee-object-id", principal_id,
             "--assignee-principal-type", "ServicePrincipal", "--role", role, "--scope", scope])

_ensure_role(_project_principal, "AcrPull", _acr_resource_id, "Foundry project MI")
try:
    _existing = project_client.agents.get(agent_name=AGENT_NAME).as_dict()
    _runtime_principal = (_existing.get("instance_identity") or {}).get("principal_id")
except Exception:
    _runtime_principal = None
if _runtime_principal:
    _ensure_role(_runtime_principal, "Cognitive Services OpenAI User", _foundry_account_id, "agent runtime MI")
    _ensure_role(_runtime_principal, "Foundry User", _foundry_account_id, "agent runtime MI")

# ── Step 3.6: Ensure App Insights for telemetry ──────────────
print("\n▶ Step 3.6: Ensuring Application Insights resource...")

def _ensure_app_insights_single() -> tuple[str, str]:
    appi_name = f"{ACCOUNT_NAME}-appi"
    law_name = f"{ACCOUNT_NAME}-law"
    location = _run_az(["group", "show", "-n", RESOURCE_GROUP, "--query", "location", "-o", "tsv"]) or "eastus"
    for q in ("properties.applicationInsightsResourceId", "properties.applicationInsightsId",
              "properties.applicationInsights", "properties.monitoring.applicationInsightsResourceId"):
        rid = _run_az(["cognitiveservices", "account", "show", "--name", ACCOUNT_NAME,
                       "--resource-group", RESOURCE_GROUP, "--query", q, "-o", "tsv"])
        if rid and rid != "None" and rid.startswith("/subscriptions/"):
            conn = _run_az(["monitor", "app-insights", "component", "show", "--ids", rid,
                            "--query", "connectionString", "-o", "tsv"])
            return rid, conn
    rid = _run_az(["resource", "list", "--resource-group", RESOURCE_GROUP,
                   "--resource-type", "microsoft.insights/components", "--query", "[0].id", "-o", "tsv"])
    if rid and rid != "None":
        conn = _run_az(["monitor", "app-insights", "component", "show", "--ids", rid,
                        "--query", "connectionString", "-o", "tsv"])
        return rid, conn
    law_id = _run_az(["monitor", "log-analytics", "workspace", "show", "-g", RESOURCE_GROUP,
                      "-n", law_name, "--query", "id", "-o", "tsv"])
    if not law_id:
        print(f"  • Creating Log Analytics workspace '{law_name}'...")
        law_id = _run_az(["monitor", "log-analytics", "workspace", "create", "-g", RESOURCE_GROUP,
                          "-n", law_name, "--location", location, "--query", "id", "-o", "tsv"])
    print(f"  • Creating Application Insights '{appi_name}'...")
    _run_az(["monitor", "app-insights", "component", "create", "-g", RESOURCE_GROUP, "-a", appi_name,
             "--location", location, "--workspace", law_id, "--query", "id", "-o", "tsv"])
    appi_id = _run_az(["resource", "list", "--resource-group", RESOURCE_GROUP,
                       "--resource-type", "microsoft.insights/components", "--query", "[0].id", "-o", "tsv"])
    conn = _run_az(["monitor", "app-insights", "component", "show", "-g", RESOURCE_GROUP,
                    "-a", appi_name, "--query", "connectionString", "-o", "tsv"])
    return appi_id, conn

_appi_resource_id, _appi_connection_string = _ensure_app_insights_single()
if _appi_resource_id:
    print(f"  ✓ App Insights: {_appi_resource_id.split('/')[-1]}")

print("\n▶ Step 4: Creating Hosted Agent version...")
agent_version = project_client.agents.create_version(
    agent_name=AGENT_NAME,
    definition=HostedAgentDefinition(
        container_protocol_versions=[ProtocolVersionRecord(protocol=AgentProtocol.RESPONSES, version="1.0.0")],
        cpu="1", memory="2Gi", image=FULL_IMAGE,
        environment_variables={"MODEL_DEPLOYMENT_NAME": MODEL_DEPLOYMENT_NAME},
    ),
)
print(f"  Agent: {agent_version.name}, Version: {agent_version.version}")

try:
    _doc = project_client.agents.get(agent_name=AGENT_NAME).as_dict()
    _runtime_principal = (_doc.get("instance_identity") or {}).get("principal_id")
    if _runtime_principal:
        _ensure_role(_runtime_principal, "Cognitive Services OpenAI User", _foundry_account_id, "agent runtime MI")
        _ensure_role(_runtime_principal, "Foundry User", _foundry_account_id, "agent runtime MI")
        if _appi_resource_id:
            _ensure_role(_runtime_principal, "Monitoring Metrics Publisher", _appi_resource_id, "agent runtime MI")
        print("  ℹ️  Runtime RBAC propagation can take 30–90s before the agent can call models.")
except Exception as exc:
    print(f"  ⚠️  Could not resolve runtime identity after create_version: {exc}")

print("\n▶ Step 5: Waiting for agent to become active...")
while True:
    info = project_client.agents.get_version(agent_name=AGENT_NAME, agent_version=agent_version.version)
    status = info["status"]
    print(f"  Status: {status}")
    if status == "active":
        print("  ✅ Agent is ready!"); break
    elif status == "failed":
        print(f"  ❌ Provisioning failed: {info.get('error')}"); break
    await asyncio.sleep(5)


▶ Step 1: Discovering Azure Container Registry...
  ACR:   htamlcr.azurecr.io
  Image: htamlcr.azurecr.io/impossible-travel-skill-agent:v1

▶ Step 2: Building Docker image...

▶ Step 3: Pushing to ACR...
  ✅ Image pushed

▶ Step 3.5: Ensuring required RBAC role assignments...
  ✓ 'AcrPull' already granted to Foundry project MI
  ✓ 'Cognitive Services OpenAI User' already granted to agent runtime MI
  ✓ 'Foundry User' already granted to agent runtime MI

▶ Step 3.6: Ensuring Application Insights resource...
  ✓ App Insights: fndryiac2ttkx3-appi

▶ Step 4: Creating Hosted Agent version...
  Agent: impossible-travel-skill-agent, Version: 3
  ✓ 'Cognitive Services OpenAI User' already granted to agent runtime MI
  ✓ 'Foundry User' already granted to agent runtime MI
  • Granting 'Monitoring Metrics Publisher' to agent runtime MI on fndryiac2ttkx3-appi
  ℹ️  Runtime RBAC propagation can take 30–90s before the agent can call models.

▶ Step 5: Waiting for agent to become active...
  Status: 

## Invoke the Deployed Agent

In [4]:
from pydantic import BaseModel, Field

class ActionItem(BaseModel):
    action: str; target: str = ""; value: str = ""; riskFactor: str = ""
    destructive: bool = False; requiresApproval: bool = False

class InvestigationVerdict(BaseModel):
    verdict: str; confidence: str; reasoning: str
    actionPlan: list[ActionItem] = Field(default_factory=list)
    narrative: str = ""

TEST_CASE = "03_mfa_fatigue_attack"
case_path = Path("test_cases") / f"{TEST_CASE}.json"
with open(case_path, encoding="utf-8") as f:
    test_case_data = json.load(f)
detection_payload = test_case_data["detection"]
incident_json = json.dumps(detection_payload)

print(f"▶ Invoking deployed agent with: {test_case_data['name']}")
print(f"   User:     {detection_payload['PrimaryEntityId']}")
print(f"   Expected: {test_case_data.get('expected_verdict', '—')} ({test_case_data.get('expected_confidence', '—')})\n")

openai_client = project_client.get_openai_client(agent_name=AGENT_NAME)
response = openai_client.responses.create(input=incident_json)

print("═" * 70)
print("📋 HOSTED AGENT RESPONSE")
print("═" * 70)
print(response.output_text)
try:
    hosted_verdict = InvestigationVerdict.model_validate_json(response.output_text)
    print(f"\n  Verdict:    {hosted_verdict.verdict}")
    print(f"  Confidence: {hosted_verdict.confidence}")
    print(f"  Actions:    {len(hosted_verdict.actionPlan)}")
except Exception:
    print("  (Could not parse structured verdict from response)")


▶ Invoking deployed agent with: MFA Fatigue Attack
   User:     bob.chen@contoso.com
   Expected: AccountCompromised (High)

══════════════════════════════════════════════════════════════════════
📋 HOSTED AGENT RESPONSE
══════════════════════════════════════════════════════════════════════
{
  "verdict": "Compromised",
  "confidence": "High",
  "reasoning": "The user bob.chen@contoso.com exhibits multiple risk factors: sign-ins from geographically distant locations (Seattle and Sofia) in a short timeframe with suspicious IP reputation and proxy use; concurrent sessions from different IPs; rapid succession of failed sign-in attempts followed by success from the suspicious IP indicating brute-force attempts; MFA method recently changed immediately after suspicious login with indications of MFA fatigue through multiple denied then accepted MFA prompts; suspicious app access to Microsoft Graph and SharePoint Online from the suspicious IP; and overall, the account is marked at risk with act

## Token Usage Report

Query Application Insights for **actual** per-agent token consumption.
Run after invoking the deployed agent. Waits briefly for ingestion.


In [19]:
# Token usage report — self-contained; works after kernel restart.

import subprocess as _subprocess, sys as _sys, time as _time, json as _json
from pathlib import Path as _Path

if 'RESOURCE_GROUP' not in dir():
    _cfg_path = _Path('../workshop_config.json')
    with open(_cfg_path) as _f: _cfg = _json.load(_f)
    RESOURCE_GROUP = _cfg['RESOURCE_GROUP']
    ACCOUNT_NAME = _cfg['ACCOUNT_NAME']

_USE_SHELL_TOKEN = _sys.platform == 'win32'
def _run_az_token(args):
    r = _subprocess.run(['az', *args], capture_output=True, text=True, shell=_USE_SHELL_TOKEN)
    return r.stdout.strip() if r.returncode == 0 else ''

_appi_rid = _run_az_token(['resource', 'list', '--resource-group', RESOURCE_GROUP,
    '--resource-type', 'microsoft.insights/components', '--query', '[0].id', '-o', 'tsv'])
_appi_name = _appi_rid.split('/')[-1] if _appi_rid and _appi_rid != 'None' else ''

INGESTION_WAIT_S = 45
LOOKBACK_MINUTES = 30
AGENT_FILTER = 'impossible-travel-skill-agent'  # must match AGENT_NAME in the deploy cell
print(f'App Insights: {_appi_name or "(not found)"}')
print(f'Agent filter: {AGENT_FILTER}')
print(f'Waiting {INGESTION_WAIT_S}s for ingestion...')
_time.sleep(INGESTION_WAIT_S)

_kql = (
    'dependencies'
    '| where timestamp > ago(' + str(LOOKBACK_MINUTES) + 'm)'
    '| where name startswith "chat "'
    '| extend agent_id = tostring(coalesce(customDimensions["gen_ai.agent.name"], customDimensions["gen_ai.agent.id"]))'
    '| where agent_id has "' + AGENT_FILTER + '"'
    '| extend'
    '    agent  = tostring(coalesce(customDimensions["gen_ai.agent.name"], customDimensions["gen_ai.agent.id"], name)),'
    '    in_t   = toint(coalesce(customDimensions["gen_ai.usage.input_tokens"],  customDimensions["gen_ai.usage.prompt_tokens"], customDimensions["llm.usage.prompt_tokens"])),'
    '    out_t  = toint(coalesce(customDimensions["gen_ai.usage.output_tokens"], customDimensions["gen_ai.usage.completion_tokens"], customDimensions["llm.usage.completion_tokens"])),'
    '    model  = tostring(coalesce(customDimensions["gen_ai.response.model"],   customDimensions["llm.response.model"], name))'
    '| where isnotnull(in_t) or isnotnull(out_t)'
    '| summarize calls = count(),'
    '            input_tokens  = sum(in_t),'
    '            output_tokens = sum(out_t),'
    '            total_tokens  = sum(in_t) + sum(out_t)'
    '  by agent, model'
    '| order by agent asc'
)

if not _appi_name:
    print('⚠️  No App Insights found. Run the deploy cell first.')
else:
    _raw = _run_az_token(['monitor', 'app-insights', 'query', '--app', _appi_name,
        '-g', RESOURCE_GROUP, '--analytics-query', _kql, '-o', 'json'])
    if not _raw:
        print(f'⚠️  No data in the last {LOOKBACK_MINUTES} minutes.')
    else:
        _result = _json.loads(_raw)
        _tables = _result.get('tables', _result) if isinstance(_result, dict) else _result
        if isinstance(_tables, list) and len(_tables) > 0:
            _table = _tables[0] if isinstance(_tables[0], dict) else _tables
            _columns = _table.get('columns', [])
            _rows = _table.get('rows', [])
            _col_names = [c.get('name', c) if isinstance(c, dict) else str(c) for c in _columns]
            print(f"{'Agent':<40} {'Model':<18} {'Calls':>5} {'Input':>8} {'Output':>8} {'Total':>8}")
            print('─' * 96)
            gi = go = gt = gc = 0
            for row in _rows:
                a = str(row[_col_names.index('agent')])
                m = str(row[_col_names.index('model')])
                c = int(row[_col_names.index('calls')])
                i = int(row[_col_names.index('input_tokens')])
                o = int(row[_col_names.index('output_tokens')])
                t = int(row[_col_names.index('total_tokens')])
                print(f'{a:<40} {m:<18} {c:>5} {i:>8} {o:>8} {t:>8}')
                gi += i; go += o; gt += t; gc += c
            print('─' * 96)
            print(f"{'TOTAL':<40} {'':<18} {gc:>5} {gi:>8} {go:>8} {gt:>8}")
        else:
            print('⚠️  Unexpected result shape:', _json.dumps(_result, indent=2)[:1000])


App Insights: fndryiac2ttkx3-appi
Agent filter: impossible-travel-skill-agent
Waiting 45s for ingestion...
Agent                                    Model              Calls    Input   Output    Total
────────────────────────────────────────────────────────────────────────────────────────────────
impossible-travel-skill-agent            gpt-4.1-mini-2025-04-14     4    10558     1663    12221
────────────────────────────────────────────────────────────────────────────────────────────────
TOTAL                                                           4    10558     1663    12221


In [6]:
# Cleanup (Optional)
# print("▶ Cleaning up hosted agent...")
# project_client.agents.delete(agent_name=AGENT_NAME)
# print("✅ Agent deleted")


## Notes

- The deployed container ships the **same** `SKILL.md`, references, scripts, tools, and test cases used locally — 
deployment is a pure packaging step.
- In production, swap `tools.py` mocks for MCP calls to Sentinel / Entra / Defender XDR. The agent and the skill stay unchanged.
- Compare cost / latency / accuracy with the multi-agent deployment on `main` using the JSONL output of `01-investigation.ipynb`.
